# Lab 2: Vectorless RAG — Advanced Scenarios

This lab covers multi-hop aggregation and structured data fidelity. See `lab2_vectorless_rag_advanced.md` for full writeup and architecture diagrams.

## Setup

### Install Dependencies

Install the required Python packages: `pageindex` for document tree generation, `openai` as the LLM client for OpenRouter, `python-dotenv` for loading API keys, and `pymupdf` for PDF text extraction.

In [ ]:
!pip install -q pageindex openai python-dotenv pymupdf

### Import Libraries

Import the standard library and third-party modules used throughout the notebook.

In [ ]:
import os
import json
import re
import fitz
from openai import OpenAI
from dotenv import load_dotenv

### Load API Keys

Load API keys from the `.env` file in the project root (`../.env` relative to this notebook). If either key is missing from `.env`, you'll be prompted to enter it manually.

In [ ]:
load_dotenv("../.env")

PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

# If keys are missing, prompt the user to enter them
if not PAGEINDEX_API_KEY:
    PAGEINDEX_API_KEY = input("Enter your PageIndex API key (get one at https://pageindex.ai): ").strip()
if not OPENROUTER_API_KEY:
    OPENROUTER_API_KEY = input("Enter your OpenRouter API key (get one at https://openrouter.ai): ").strip()

print("Keys loaded.")

### Set up the LLM

The `call_llm` function sends a prompt to the LLM via OpenRouter and returns the response. It uses `meta-llama/llama-4-scout-17b-16e-instruct` by default.

In [ ]:
def call_llm(prompt, model="meta-llama/llama-4-scout-17b-16e-instruct"):
    """Call a language model via OpenRouter."""
    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=OPENROUTER_API_KEY,
    )
    msgs = [{"role": "user", "content": prompt}]
    resp = client.chat.completions.create(model=model, messages=msgs, temperature=0, max_tokens=1024)
    return resp.choices[0].message.content.strip()

---
## Load and Parse the PDF

Extract text from each page of the PDF using PyMuPDF. This text will be used as context for the LLM when answering questions.

### Define PDF Path

Point this to any PDF you want to query.

In [ ]:
PDF_PATH = "data/your_document.pdf"

### Extract Text from PDF

Opens the PDF with PyMuPDF and builds a dictionary mapping 1-based page numbers to their extracted text.

In [ ]:
doc = fitz.open(PDF_PATH)
page_texts = {i+1: doc.load_page(i).get_text() for i in range(len(doc))}
doc.close()
print(f"Extracted text from {len(page_texts)} pages.")

---
## Build Document Tree (with caching)

The PageIndex API parses the PDF into a hierarchical tree of sections and subsections. The tree is cached as JSON so repeat runs with the same PDF skip the API call.

### Set Up Caching and PageIndex

Import the PageIndex client and define the cache path. The cache file is named after the PDF filename with `_tree.json` appended.

In [ ]:
from pageindex import PageIndexClient
from pageindex import utils
import time

CACHE_PATH = f"cache/{os.path.basename(PDF_PATH).replace('.pdf', '_tree.json')}"
os.makedirs("cache", exist_ok=True)

### Load or Build the Tree

Try loading from cache first. If it's a cache miss, submit the PDF to PageIndex, poll until processing completes (up to 5 minutes), save the result to cache, and display the tree structure.

In [ ]:
tree = None
if os.path.exists(CACHE_PATH):
    with open(CACHE_PATH) as f:
        tree = json.load(f)
    print("Loaded tree from cache.")

if tree is None:
    pi = PageIndexClient(api_key=PAGEINDEX_API_KEY)
    result = pi.submit_document(PDF_PATH)
    doc_id = result["doc_id"]
    print(f"Submitted: {doc_id}")

    elapsed = 0
    while elapsed < 300:
        if pi.is_retrieval_ready(doc_id):
            break
        time.sleep(5)
        elapsed += 5
        print(f"  {elapsed}s...")
    else:
        raise TimeoutError("PageIndex timeout")

    tree = pi.get_tree(doc_id, node_summary=True)["result"]
    with open(CACHE_PATH, "w") as f:
        json.dump(tree, f, indent=2)
    print("Tree cached.")

utils.print_tree(tree, exclude_fields=["text"])

---
## Helper Functions

Reusable functions for retrieving nodes and building context from the document tree.

### Parse JSON

Extracts a JSON object from the LLM's response text. Handles markdown code fences (```` ```json ... ``` ````) that LLMs often wrap around JSON output.

In [ ]:
def parse_json(text):
    """Extract JSON from LLM response."""
    text = re.sub(r"```json\s*|\s*```", "", text.strip())
    s, e = text.find("{"), text.rfind("}")
    if s != -1 and e != -1:
        text = text[s:e+1]
    return json.loads(text)

### Retrieve Nodes

Uses the LLM to find relevant nodes for a given query. Strips full text from the tree (keeping only titles + summaries), sends it to the LLM with the question, and returns the parsed JSON result containing the LLM's reasoning and a list of relevant node IDs.

In [ ]:
def retrieve_nodes(query):
    """Use LLM to find relevant nodes for a query."""
    tree_slim = utils.remove_fields(tree.copy(), fields=["text"])
    prompt = f"""
Given a question and a document tree (node_id, title, summary),
find nodes likely to contain the answer.

Question: {query}

Tree:
{json.dumps(tree_slim, indent=2)}

JSON only:
{{"thinking": "...", "node_list": ["id1", "id2"]}}
"""
    return parse_json(call_llm(prompt))

### Get Context

Extracts text from pages covered by the given node list. Maps each node ID to its page range, collects the extracted text from those pages (deduplicating), and joins them with page separators.

In [ ]:
def get_context(node_list):
    """Extract text from pages covered by the given nodes."""
    node_map = utils.create_node_mapping(tree, include_page_ranges=True, max_page=len(page_texts))
    texts, seen = [], set()
    for nid in node_list:
        info = node_map[nid]
        for p in range(info["start_index"], info["end_index"] + 1):
            if p not in seen and p in page_texts:
                texts.append(f"--- Page {p} ---\n{page_texts[p]}")
                seen.add(p)
    return "\n\n".join(texts)

---
# Scenario 1: Multi-Hop Attribute Aggregation

**Problem:** Some questions require combining information from multiple sections. For example:
- "What is the total cost including tax?" (requires item price + tax rate)
- "What are the requirements and deadlines?" (requires specs + schedule)

**Solution:** The LLM must retrieve nodes from multiple sections and aggregate the information.

### Define Multi-Hop Query

Set a question that requires information from multiple document sections to answer.

In [ ]:
# Change this to a question that requires info from multiple sections
QUERY_MULTI_HOP = "What is the total cost including all fees and taxes?"

### Retrieve Nodes for Multi-Hop Query

Send the query to the LLM to identify which nodes are relevant. For multi-hop questions, the LLM should return nodes from multiple sections.

In [ ]:
result = retrieve_nodes(QUERY_MULTI_HOP)
print("Reasoning:", result["thinking"], "\n")
print("Retrieved nodes:", result["node_list"])

### Answer Multi-Hop Query

Build context from the retrieved nodes and send it to the LLM with the question to get the final answer.

In [ ]:
context = get_context(result["node_list"])
answer = call_llm(f"Context:\n{context}\n\nQuestion: {QUERY_MULTI_HOP}\n\nAnswer concisely.")
print("Answer:", answer)

### Why Multi-Hop is Hard

Traditional RAG retrieves chunks by similarity. A query about "total cost" might only match the pricing section OR the tax section — not both. Vectorless RAG with tree-based reasoning can identify that the answer requires **multiple sections**.

---
# Scenario 2: Structured Data Fidelity

**Problem:** Documents contain tables, forms, and structured data. Extracting this data accurately is critical — a wrong number can be costly.

**Solution:** The LLM reads the raw text (which preserves table structure) and extracts values with high fidelity.

### Define Structured Data Query

Set a question about extracting specific values from a table in the document.

In [ ]:
# Change this to a question about extracting specific values from a table
QUERY_STRUCTURED = "What are the limits or caps mentioned in the table?"

### Retrieve Nodes for Structured Data

Find the nodes containing the relevant table or structured data.

In [ ]:
result = retrieve_nodes(QUERY_STRUCTURED)
print("Reasoning:", result["thinking"], "\n")
print("Retrieved nodes:", result["node_list"])

### Answer Structured Data Query

Extract the page text from the retrieved nodes and have the LLM read the raw table data to produce an accurate answer.

In [ ]:
context = get_context(result["node_list"])
answer = call_llm(f"Context:\n{context}\n\nQuestion: {QUERY_STRUCTURED}\n\nAnswer concisely.")
print("Answer:", answer)

### Why Structured Data Fidelity Matters

Tables in PDFs are often poorly formatted when extracted. Vectorless RAG preserves the original text flow, allowing the LLM to understand table structure from context (column headers, row labels, etc.).

---
# Scenario 3: Combined — Multi-Hop + Structured Data

The hardest case: a question that requires **both** multi-hop reasoning AND accurate structured data extraction.

### Define Combined Query

Set a question that needs both multi-hop reasoning across sections and accurate table extraction.

In [ ]:
# Change this to a question that needs both multi-hop and table extraction
QUERY_COMBINED = "What is the final amount after applying all adjustments from the tables?"

### Retrieve Nodes for Combined Query

Find nodes from multiple sections that contain both the table data and the related calculations.

In [ ]:
result = retrieve_nodes(QUERY_COMBINED)
print("Reasoning:", result["thinking"], "\n")
print("Retrieved nodes:", result["node_list"])

### Answer Combined Query

Build context from all relevant nodes and have the LLM aggregate information from tables while performing multi-hop reasoning.

In [ ]:
context = get_context(result["node_list"])
answer = call_llm(f"Context:\n{context}\n\nQuestion: {QUERY_COMBINED}\n\nAnswer concisely.")
print("Answer:", answer)

---
## Try It Yourself

Change the `QUERY_*` variables and re-run the cells. Here are some generic patterns to try:

In [ ]:
examples = [
    # Multi-hop
    "What are the requirements and how do they change over time?",
    "What are the sub-components and their respective limits?",
    
    # Structured data
    "List all items and their corresponding values from the table.",
    "What is the breakdown of costs by category?",
    
    # Combined
    "Calculate the total based on all the tables and sections.",
]

for q in examples:
    result = retrieve_nodes(q)
    print(f"{q[:50]}... -> {len(result['node_list'])} nodes")